In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
# --------------------- Import VQNiche ---------------------
from vqniche.utils.parse_test_configs import *
from vqniche.initializers.initialize import *
from vqniche.utils.type_conversions import *
from vqniche.plotting import *

/software/cellgen/team361/am84/envs/vqniche-reproducibility/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/software/cellgen/team361/am84/envs/vqniche-reproducibility/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/software/cellgen/team361/am84/envs/vqniche-reproducibility/lib/python3.10/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain

In [3]:
# --------------------- Import Libraries ---------------------
import re
import os
import copy
import sys
import yaml
import pickle
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional
import statistics

import scanpy as sc
import anndata as ad
import squidpy as sq

import numpy as np
import networkx as nx
import scipy.sparse as sp
from scipy.stats import pearsonr

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import pytorch_lightning as pl
import torch_geometric.transforms as T
from torch_geometric.data import Batch
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_dense_adj
from torch_geometric.loader import DataLoader as BatchBuilder

# --------------------- Display Settings ---------------------
# display setting all rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [4]:
save_dir = Path("/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/paper/tables")
save_dir.mkdir(parents=True, exist_ok=True)

In [8]:
def parse_timestamp(time_str: str) -> Optional[datetime]:
    """
    Parse ISO format timestamp, handling various fractional second precisions.
    
    Args:
        time_str: ISO format timestamp string
        
    Returns:
        datetime object or None if parsing fails
    """
    try:
        # Handle fractional seconds with more than 6 digits
        # Pattern: YYYY-MM-DDTHH:MM:SS.ffffff+HH:MM or ...Z
        pattern = r'(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2})\.(\d+)([\+\-Z].*)'
        match = re.match(pattern, time_str)
        
        if match:
            date_part = match.group(1)
            fractional = match.group(2)
            timezone_part = match.group(3)
            
            # Truncate fractional seconds to 6 digits max
            fractional = fractional[:6]
            
            # Handle 'Z' as '+00:00'
            if timezone_part == 'Z':
                timezone_part = '+00:00'
            
            # Reconstruct the timestamp
            time_str = f"{date_part}.{fractional}{timezone_part}"
        
        return datetime.fromisoformat(time_str)
        
    except Exception as e:
        return None

    
def parse_debug_internal_log(log_file_path: Path) -> Optional[float]:
    """
    Parse a wandb debug-internal.log file to extract elapsed time.
    
    Args:
        log_file_path: Path to debug-internal.log file
        
    Returns:
        Elapsed time in seconds, or None if parsing fails
    """
    try:
        with open(log_file_path, 'r') as f:
            lines = f.readlines()
        
        start_time = None
        stop_time = None
        
        for line in lines:
            try:
                log_entry = json.loads(line.strip())
                
                # Look for system monitor start
                if log_entry.get('msg') == 'Starting system monitor':
                    start_time = parse_timestamp(log_entry['time'])
                
                # Look for system monitor stop
                elif log_entry.get('msg') == 'Stopping system monitor':
                    stop_time = parse_timestamp(log_entry['time'])
                    
            except (json.JSONDecodeError, KeyError, ValueError):
                continue
        
        if start_time and stop_time:
            elapsed = (stop_time - start_time).total_seconds()
            return elapsed
        
        return None
        
    except Exception as e:
        print(f"Error parsing {log_file_path}: {e}")
        return None


def analyze_sweep_elapsed_times(sweep_path: str) -> Dict[str, any]:
    """
    Analyze elapsed times for all runs in a sweep directory.
    
    Args:
        sweep_path: Path to sweep directory (e.g., 
            /path/to/logs/.../seed/20250918-120525)
            
    Returns:
        Dictionary with statistics:
        - run_times: List of individual run times in seconds
        - run_times_min: List of individual run times in minutes
        - mean_time_sec: Mean elapsed time in seconds
        - mean_time_min: Mean elapsed time in minutes
        - median_time_sec: Median elapsed time in seconds
        - median_time_min: Median elapsed time in minutes
        - min_time_sec: Minimum elapsed time
        - max_time_sec: Maximum elapsed time
        - num_runs: Number of runs analyzed
        - run_details: List of dicts with run name and time
    """
    sweep_dir = Path(sweep_path)
    
    if not sweep_dir.exists():
        print(f"Directory not found: {sweep_path}")
        return {}
    
    # Find wandb directory
    wandb_dir = sweep_dir / "wandb"
    if not wandb_dir.exists():
        print(f"No wandb directory found in {sweep_path}")
        return {}
    
    # Find all run directories (both online and offline)
    run_dirs = []
    for pattern in ['run-*', 'offline-run-*']:
        run_dirs.extend(wandb_dir.glob(pattern))
    
    if not run_dirs:
        print(f"No run directories found in {wandb_dir}")
        return {}
    
    # Parse each run's debug-internal.log
    run_times = []
    run_details = []
    
    for run_dir in sorted(run_dirs):
        log_file = run_dir / "logs" / "debug-internal.log"
        
        if not log_file.exists():
            continue
        
        elapsed = parse_debug_internal_log(log_file)
        
        if elapsed is not None:
            run_times.append(elapsed)
            run_details.append({
                'run_name': run_dir.name,
                'elapsed_sec': elapsed,
                'elapsed_min': elapsed / 60
            })
    
    if not run_times:
        print(f"No valid timing data extracted from {wandb_dir}")
        return {}
    
    # Calculate statistics
    meantime = statistics.mean(run_times)
    meantime_min = int(meantime/60)
    meantime_sec = int(meantime - meantime_min*60)
    meantime_str = f"{mins} minutes {secs} seconds"
    results = {
        'num_runs': len(run_times),
        'run_times': run_times,
        'run_times_min': [t / 60 for t in run_times],
        'mean_time_sec': statistics.mean(run_times),
        'mean_time_min': statistics.mean(run_times) / 60,
        'median_time_sec': statistics.median(run_times),
        'median_time_min': statistics.median(run_times) / 60,
        'min_time_sec': min(run_times),
        'max_time_sec': max(run_times),
        'run_details': run_details,
        'mean_time':  meantime_str
    }
    
    # Add standard deviation if we have more than one run
    if len(run_times) > 1:
        results['std_dev_sec'] = statistics.stdev(run_times)
        results['std_dev_min'] = results['std_dev_sec'] / 60
    
    return results


def get_peak_memory_from_job_log(log_file_path: str) -> Optional[float]:
    """
    Extract peak memory usage from an LSF job log file.
    
    Args:
        log_file_path: Path to the LSF job log .out file
        
    Returns:
        Peak memory in GB, or None if not found
    """
    try:
        with open(log_file_path, 'r') as f:
            content = f.read()
        
        # Look for "Max Memory : XXXX MB" pattern
        match = re.search(r'Max Memory\s*:\s*([\d.]+)\s*MB', content)
        
        if match:
            memory_mb = float(match.group(1))
            memory_gb = memory_mb / 1024
            memory_str = f"{memory_gb:.2f} GB"
        
        # Look for "Run time : XXX sec." pattern
        match = re.search(r'Run time\s*:\s*([\d.]+)\s*sec', content)
        
        if match:
            runtime_sec = float(match.group(1))/3
            meantime = np.mean(runtime_sec)
            meantime_min = int(meantime/60)
            meantime_sec = int(meantime - meantime_min*60)
            meantime_str = f"{meantime_min} minutes {meantime_sec} seconds"
            print(meantime_str)
        
        return memory_str, meantime_str
        
    except Exception as e:
        print(f"Error reading {log_file_path}: {e}")
        return None

In [9]:
mmb0_sweep_dir = Path("/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/logs/mmb0-4b_1p/sweep/VQNiche/batch=[0, 1, 2, 3]/spatial_n_neighs_8/seed/20250918-120525")
mmb0_job_log = Path("/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/logs/mmb0-4b_1p/sweep/job_log/vqniche_graphsage/seed/6_gpu-lotfollahi-train_888928.out")

xhk1020_sweep_dir = Path("/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/logs/xhk1020-CV1-CV2-5b_1p/sweep/VQNiche/batch=[0, 1, 2, 3, 4]/spatial_n_neighs_8/seed/20250918-133207")
xhk1020_job_log = Path("/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/logs/xhk1020-CV1-CV2-5b_1p/sweep/job_log/vqniche_graphsage/seed/6_gpu-lotfollahi-train_916220.out")

xhs1000_sweep_dir = Path("/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/logs/xhs1000-39b_1p/sweep/VQNiche/batch=[2, 11, 9, 28, 29, 32, 12]/spatial_n_neighs_8/seed/20250923-220522")
xhs1000_job_log = Path("/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/logs/xhs1000-39b_1p-oriented-7/sweep/job_log/vqniche_graphsage/seed/6_gpu-normal_605307.out")


In [10]:
datasets = ['Brain (Mouse)', 'Kidney (Human)', 'Skin (Human)']
sweep_dirs = [mmb0_sweep_dir, xhk1020_sweep_dir, xhs1000_sweep_dir]
job_logs = [mmb0_job_log, xhk1020_job_log, xhs1000_job_log]

stats = []
for i, dataset in enumerate(datasets):
    peak_memory, avg_runtime = get_peak_memory_from_job_log(job_logs[i])
    stats.append([dataset, peak_memory, avg_runtime])

2 minutes 52 seconds
16 minutes 53 seconds
6 minutes 1 seconds


In [11]:
df = pd.DataFrame(stats, columns=['Dataset', 'Peak Memory', 'Runtime'])

In [14]:
df

,Dataset,Peak Memory,Runtime
0,Brain (Mouse),4.17 GB,2 minutes 52 seconds
1,Kidney (Human),15.83 GB,16 minutes 53 seconds
2,Skin (Human),16.46 GB,6 minutes 1 seconds


In [15]:
print(df.to_markdown())

|    | Dataset        | Peak Memory   | Runtime               |
|---:|:---------------|:--------------|:----------------------|
|  0 | Brain (Mouse)  | 4.17 GB       | 2 minutes 52 seconds  |
|  1 | Kidney (Human) | 15.83 GB      | 16 minutes 53 seconds |
|  2 | Skin (Human)   | 16.46 GB      | 6 minutes 1 seconds   |
